# calibration: camera intrinsics from a ChArUco board (M1)

A ChArUco board is a chessboard with ArUco markers in the white squares. The markers make
every inner chessboard corner identifiable even when the board is partly out of view, so
each photo gives a set of (3D board point, 2D image point) pairs. `cv2.calibrateCamera`
then fits the camera matrix K (fx, fy, cx, cy) and the lens distortion coefficients by
minimizing the reprojection error over all views.

Output: `calibration/camera_intrinsics.yaml` (path from `configs/camera.yaml:
intrinsics_file`), used to undistort frames before marker detection (`homography.ipynb`).
The board itself is printed with `markers.ipynb`; photos are taken in `run_calibrate.ipynb`.

Milestone: M1. Deliverable: intrinsics + reprojection error report.

**TODO**
- [ ] `make_charuco_board`, `detect_charuco`
- [ ] `reprojection_errors`, `calibrate_from_points`, `calibrate_charuco`
- [ ] `save_intrinsics`, `load_intrinsics`, `undistort`
- [ ] `collect_views`: live capture loop (show detections, keep a frame on keypress, reject
      blurry views and views with < min_corners corners)
- [ ] `coverage_report`: check the views cover the whole image (corners and edges) and a
      range of tilts; poor coverage -> unreliable distortion estimate
- [ ] decide on the distortion model (default 5 coefficients; try `CALIB_RATIONAL_MODEL`
      only if the error stays high) and an acceptance threshold (e.g. RMS < 0.5 px)
- [ ] write the self-check (plan at the bottom)

In [ ]:
from __future__ import annotations

import datetime as _dt
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import yaml

%run ./config.ipynb

## The ChArUco board
Geometry from `configs/camera.yaml: charuco` (squares, square/marker length in meters,
dictionary). Measure the printed square length and update the config, because the scale
of the 3D points comes from it (it affects the extrinsics, not K).

In [ ]:
def make_charuco_board(charuco_cfg=None):
    """cv2.aruco.CharucoBoard from the charuco section of camera.yaml.

    Inputs: charuco_cfg namespace (squares_x, squares_y, square_length, marker_length,
    dictionary); None -> load_config("configs/camera.yaml").charuco.
    """
    # Design:
    # - Dictionary from its name: cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, name)).
    # - cv2.aruco.CharucoBoard((squares_x, squares_y), square_length, marker_length, dictionary).
    # - Import cv2 inside the function (keeps notebooks importable without OpenCV).
    # TODO: implement
    raise NotImplementedError


def detect_charuco(image: np.ndarray, board, min_corners: int = 4):
    """Detect ChArUco corners in one image.

    Returns (corners (N, 1, 2) float32, ids (N, 1) int32), or (None, None) if fewer than
    min_corners were found.
    """
    # Design:
    # - Convert to grayscale if the image is BGR.
    # - cv2.aruco.CharucoDetector(board).detectBoard(gray) returns charuco corners/ids plus
    #   the raw marker detections; only the charuco corners matter for calibration.
    # - Too few corners -> (None, None) so callers can skip the view.
    # TODO: implement
    raise NotImplementedError

## Calibration and reprojection error
The reprojection error of a view is the RMS pixel distance between the detected corners and
the board points projected with the fitted K, distortion and that view's pose. Report the
overall RMS and look for outlier views (bad detections, motion blur) to drop.

In [ ]:
@dataclass
class CalibrationResult:
    rms: float                    # overall RMS reprojection error (px), from calibrateCamera
    camera_matrix: np.ndarray     # (3, 3)
    dist_coeffs: np.ndarray       # (k,)
    image_size: tuple[int, int]   # (width, height)
    per_view_errors: list[float] = field(default_factory=list)
    rvecs: list = field(default_factory=list, repr=False)
    tvecs: list = field(default_factory=list, repr=False)


def reprojection_errors(obj_points, img_points, rvecs, tvecs, K, dist) -> list[float]:
    """Per-view RMS reprojection error in pixels."""
    # Design:
    # - For each view, project the 3D board points with cv2.projectPoints(obj, rvec, tvec, K, dist).
    # - Error = sqrt(mean over points of squared Euclidean distance to the detected 2D points).
    # - Large per-view values point at bad detections / blur: those views are candidates to drop.
    # TODO: implement
    raise NotImplementedError


def calibrate_from_points(obj_points, img_points, image_size: tuple[int, int],
                          flags: int = 0) -> CalibrationResult:
    """cv2.calibrateCamera wrapper. obj_points: list of (N, 3); img_points: list of (N, 2)."""
    # Design:
    # - Reshape to the layouts OpenCV expects ((N, 1, 3) / (N, 1, 2) float32).
    # - Require >= 3 views (ValueError otherwise); calibration is under-determined below that.
    # - cv2.calibrateCamera(obj, img, image_size, None, None, flags=flags) returns
    #   (rms, K, dist, rvecs, tvecs); fill CalibrationResult incl. reprojection_errors.
    # Decision for you: which flags (fix aspect ratio? rational model?) and why.
    # TODO: implement
    raise NotImplementedError


def calibrate_charuco(views, board, image_size: tuple[int, int], flags: int = 0) -> CalibrationResult:
    """views: list of (charuco_corners, charuco_ids) from detect_charuco (None entries skipped)."""
    # Design:
    # - board.matchImagePoints(corners, ids) turns a detection into (3D obj points, 2D img points).
    # - Skip views with too few points (calibrateCamera needs >= 4 non-collinear ones; a
    #   margin such as 6 is safer), then hand everything to calibrate_from_points.
    # TODO: implement
    raise NotImplementedError

## Saving / loading the intrinsics
Plain YAML (lists, not OpenCV FileStorage) so any notebook and the ROS side can read it.

In [ ]:
def default_intrinsics_path() -> Path:
    """gv_path(camera.yaml: intrinsics_file)."""
    # TODO: implement (one line with load_config + gv_path)
    raise NotImplementedError


def save_intrinsics(result: CalibrationResult, path: str | Path | None = None) -> Path:
    """Write the calibration as plain YAML; returns the path written.

    Suggested keys: image_size, camera_matrix, dist_coeffs, rms_reprojection_error_px,
    per_view_errors_px, n_views, date.
    """
    # Design:
    # - Convert numpy arrays to nested lists (yaml.safe_dump cannot write ndarrays).
    # - Create the parent folder; default path = default_intrinsics_path().
    # TODO: implement
    raise NotImplementedError


def load_intrinsics(path: str | Path | None = None) -> dict:
    """Returns {"K": (3, 3), "dist": (k,), "image_size": (w, h), "rms": float}."""
    # Design: inverse of save_intrinsics (lists back to float arrays).
    # TODO: implement
    raise NotImplementedError


def undistort(image: np.ndarray, K: np.ndarray, dist: np.ndarray) -> np.ndarray:
    """Undistorted image with the same K (fine for a narrow-ish lens)."""
    # Design:
    # - cv2.undistort(image, K, dist) is the simple version.
    # - For the live loop, precompute maps once with cv2.initUndistortRectifyMap and use
    #   cv2.remap per frame (much faster).
    # Decision for you: keep K, or use cv2.getOptimalNewCameraMatrix (crop vs black borders).
    # TODO: implement
    raise NotImplementedError

## Collection workflow

In [ ]:
def collect_views(camera, board, n_views: int = 25, min_corners: int = 12):
    """Interactive capture of ChArUco views.

    camera: an opened cv2.VideoCapture. Returns (frames, views) where
    views = [(corners, ids), ...].
    """
    # Design:
    # - Loop: read frame, detect_charuco, draw the detected corners
    #   (cv2.aruco.drawDetectedCornersCharuco), show it.
    # - On a key press keep the frame if it has >= min_corners corners and is sharp
    #   (variance of the Laplacian above a threshold you choose).
    # - Stop after n_views. Hint for the operator: move the board to all image corners and
    #   tilt it +-30..45 degrees; keep it still.
    # TODO: implement
    raise NotImplementedError


def coverage_report(views, image_size: tuple[int, int], grid: int = 4) -> dict:
    """Fraction of a grid x grid tiling of the image that contains detected corners,
    plus the list of empty tiles, so you know where to hold the board next."""
    # Design: bin every detected corner into its tile; report covered fraction + empty tiles.
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# 1. Detection on a clean render: board.generateImage((700, 500)) with a white margin
#    (cv2.copyMakeBorder) -> detect_charuco finds all (squares_x-1)*(squares_y-1) inner
#    corners; matchImagePoints gives that many 3D points.
# 2. Synthetic calibration with a known K (e.g. fx=600, fy=610, cx=320, cy=240): project
#    board.getChessboardCorners() with ~10 random poses (cv2.projectPoints, small rotations,
#    board 0.35-0.5 m away), add 0.1 px noise, calibrate_from_points -> fx, fy within 2 %,
#    rms < 0.5 px, 10 per-view errors.
# 3. YAML round trip in a tempfile.TemporaryDirectory: save_intrinsics then load_intrinsics
#    gives the same K, dist and image_size; undistort keeps the image shape.
# 4. calibrate_from_points with 2 views raises ValueError.
print("calibration: self-check not written yet (see plan above)")